# StemScribe BTC Chord Detection — Colab Retraining

Fine-tune the BTC (Bi-directional Transformer for Chord Recognition) model with pitch augmentation
to fix class imbalance and improve accuracy on sharp/flat chords (F#m, C#m, Bm, etc.).

**What this does:**
1. Loads the existing fine-tuned BTC checkpoint (170-class large vocabulary)
2. Loads pre-computed CQT features + chord labels from Billboard/JAAH datasets
3. Augments training data with all 12 pitch transpositions per sample
4. Balances underrepresented sharp/flat root classes
5. Retrains on GPU with early stopping
6. Evaluates against ground truth for "The Time Comes" (F#m, A, Bm, E, B, D, C#m, G)

**Prerequisites:** Run `pack_for_colab.sh` locally to create `stemscribe_btc_colab.tar.gz`, then upload to Google Drive or directly to Colab.

## Cell 1: Setup and Dependencies

In [ ]:
# ============================================================
# Cell 1: Setup — Install dependencies, mount Drive, verify GPU
# ============================================================

!pip install -q torch torchvision torchaudio librosa numpy pyyaml mir_eval matplotlib

import torch
import numpy as np
import os
import sys

# Verify GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    device = torch.device('cpu')
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Working directory
WORK_DIR = '/content/btc_training'
os.makedirs(WORK_DIR, exist_ok=True)

# Drive path for checkpoints (adjust if you put the tar.gz elsewhere)
DRIVE_DATA = '/content/drive/MyDrive/stemscribe_btc'
os.makedirs(DRIVE_DATA, exist_ok=True)

print(f'Device: {device}')
print(f'Work dir: {WORK_DIR}')
print(f'Drive data dir: {DRIVE_DATA}')
print('Setup complete.')

## Cell 2: Load Checkpoint and Training Data

In [ ]:
# ============================================================
# Cell 2: Upload/extract training data package
# ============================================================
# Option A: Upload the tar.gz from your local machine
# Option B: Copy from Google Drive if already uploaded there

import tarfile
from pathlib import Path

TAR_NAME = 'stemscribe_btc_colab.tar.gz'

# Check if already extracted
if Path(f'{WORK_DIR}/btc_finetuned_best.pt').exists():
    print('Training data already extracted, skipping.')
else:
    # Try Google Drive first
    drive_tar = f'{DRIVE_DATA}/{TAR_NAME}'
    if os.path.exists(drive_tar):
        print(f'Found {TAR_NAME} on Google Drive, extracting...')
        !cp "{drive_tar}" /content/
    else:
        # Upload directly
        print(f'{TAR_NAME} not found on Drive at {drive_tar}')
        print('Upload the tar.gz file now:')
        from google.colab import files
        uploaded = files.upload()
        if TAR_NAME not in uploaded:
            # Check if any tar.gz was uploaded
            for name in uploaded:
                if name.endswith('.tar.gz'):
                    TAR_NAME = name
                    break

    # Extract
    print(f'Extracting {TAR_NAME}...')
    with tarfile.open(f'/content/{TAR_NAME}', 'r:gz') as tar:
        tar.extractall(WORK_DIR)
    print('Extraction complete.')

# Verify contents
expected_files = ['btc_finetuned_best.pt', 'run_config.yaml']
for f in expected_files:
    path = f'{WORK_DIR}/{f}'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  {f}: {size_mb:.1f} MB')
    else:
        print(f'  WARNING: {f} not found!')

# Count feature/label files
feat_dir = f'{WORK_DIR}/features'
label_dir = f'{WORK_DIR}/labels'
if os.path.isdir(feat_dir):
    n_feat = len([f for f in os.listdir(feat_dir) if f.endswith('.npy')])
    n_lab = len([f for f in os.listdir(label_dir) if f.endswith('.lab')])
    print(f'  Features: {n_feat} files')
    print(f'  Labels: {n_lab} files')
else:
    print('  No features directory found — will need to create synthetic data.')

## Cell 3: Define BTC Model Architecture

In [ ]:
# ============================================================
# Cell 3: BTC Model Architecture (from btc_chord repo)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import yaml

# --- HParams ---
class HParams(object):
    def __init__(self, **kwargs):
        self.__dict__ = kwargs
    def add(self, **kwargs):
        self.__dict__.update(kwargs)
    def update(self, **kwargs):
        self.__dict__.update(kwargs)
        return self
    @classmethod
    def load(cls, path):
        with open(path, 'r') as f:
            return cls(**yaml.load(f, Loader=yaml.SafeLoader))

# --- Chord Vocabulary (170 classes: 12 roots x 14 qualities + N + X) ---
ROOT_LIST = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
QUALITY_LIST = ['min', 'maj', 'dim', 'aug', 'min6', 'maj6', 'min7', 'minmaj7',
                'maj7', '7', 'dim7', 'hdim7', 'sus2', 'sus4']

def idx2voca_chord():
    """Build index-to-chord mapping. 170 classes total.
    Indices 0-167: 12 roots x 14 qualities
    Index 168: X (unknown)
    Index 169: N (no chord)
    
    For each root (0-11), qualities are indexed 0-13.
    When quality index == 1 (maj), chord name is just the root (e.g. 'C' not 'C:maj').
    """
    mapping = {}
    mapping[169] = 'N'
    mapping[168] = 'X'
    for i in range(168):
        root = ROOT_LIST[i // 14]
        quality_idx = i % 14
        quality = QUALITY_LIST[quality_idx]
        if quality_idx != 1:  # quality_idx 1 = 'maj' -> just root name
            chord = root + ':' + quality
        else:
            chord = root
        mapping[i] = chord
    return mapping

IDX_TO_CHORD = idx2voca_chord()
CHORD_TO_IDX = {v: k for k, v in IDX_TO_CHORD.items()}
NUM_CHORDS = 170

print(f'Chord vocabulary: {NUM_CHORDS} classes')
print(f'Sample chords: {[IDX_TO_CHORD[i] for i in range(0, 28, 1)]}')
print(f'N={IDX_TO_CHORD[169]}, X={IDX_TO_CHORD[168]}')

# --- Transformer Building Blocks ---

def _gen_bias_mask(max_length):
    np_mask = np.triu(np.full([max_length, max_length], -np.inf), 1)
    torch_mask = torch.from_numpy(np_mask).type(torch.FloatTensor)
    return torch_mask.unsqueeze(0).unsqueeze(1)

def _gen_timing_signal(length, channels, min_timescale=1.0, max_timescale=1.0e4):
    position = np.arange(length)
    num_timescales = channels // 2
    log_timescale_increment = (
        math.log(float(max_timescale) / float(min_timescale)) /
        (float(num_timescales) - 1))
    inv_timescales = min_timescale * np.exp(
        np.arange(num_timescales).astype(float) * -log_timescale_increment)
    scaled_time = np.expand_dims(position, 1) * np.expand_dims(inv_timescales, 0)
    signal = np.concatenate([np.sin(scaled_time), np.cos(scaled_time)], axis=1)
    signal = np.pad(signal, [[0, 0], [0, channels % 2]],
                    'constant', constant_values=[0.0, 0.0])
    signal = signal.reshape([1, length, channels])
    return torch.from_numpy(signal).type(torch.FloatTensor)

class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(features))
        self.beta = nn.Parameter(torch.zeros(features))
        self.eps = eps
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

class Conv(nn.Module):
    def __init__(self, input_size, output_size, kernel_size, pad_type):
        super().__init__()
        padding = (kernel_size - 1, 0) if pad_type == 'left' else (kernel_size // 2, (kernel_size - 1) // 2)
        self.pad = nn.ConstantPad1d(padding, 0)
        self.conv = nn.Conv1d(input_size, output_size, kernel_size=kernel_size, padding=0)
    def forward(self, inputs):
        inputs = self.pad(inputs.permute(0, 2, 1))
        return self.conv(inputs).permute(0, 2, 1)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, input_depth, filter_size, output_depth, layer_config='ll', padding='left', dropout=0.0):
        super().__init__()
        layers = []
        sizes = ([(input_depth, filter_size)] +
                 [(filter_size, filter_size)] * (len(layer_config) - 2) +
                 [(filter_size, output_depth)])
        for lc, s in zip(list(layer_config), sizes):
            if lc == 'l':
                layers.append(nn.Linear(*s))
            elif lc == 'c':
                layers.append(Conv(*s, kernel_size=3, pad_type=padding))
        self.layers = nn.ModuleList(layers)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
    def forward(self, inputs):
        x = inputs
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers):
                x = self.relu(x)
                x = self.dropout(x)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, input_depth, total_key_depth, total_value_depth, output_depth,
                 num_heads, bias_mask=None, dropout=0.0, attention_map=False):
        super().__init__()
        self.attention_map = attention_map
        self.num_heads = num_heads
        self.query_scale = (total_key_depth // num_heads) ** -0.5
        self.bias_mask = bias_mask
        self.query_linear = nn.Linear(input_depth, total_key_depth, bias=False)
        self.key_linear = nn.Linear(input_depth, total_key_depth, bias=False)
        self.value_linear = nn.Linear(input_depth, total_value_depth, bias=False)
        self.output_linear = nn.Linear(total_value_depth, output_depth, bias=False)
        self.dropout = nn.Dropout(dropout)
    def _split_heads(self, x):
        shape = x.shape
        return x.view(shape[0], shape[1], self.num_heads, shape[2] // self.num_heads).permute(0, 2, 1, 3)
    def _merge_heads(self, x):
        shape = x.shape
        return x.permute(0, 2, 1, 3).contiguous().view(shape[0], shape[2], shape[3] * self.num_heads)
    def forward(self, queries, keys, values):
        queries = self.query_linear(queries)
        keys = self.key_linear(keys)
        values = self.value_linear(values)
        queries = self._split_heads(queries)
        keys = self._split_heads(keys)
        values = self._split_heads(values)
        queries *= self.query_scale
        logits = torch.matmul(queries, keys.permute(0, 1, 3, 2))
        if self.bias_mask is not None:
            logits += self.bias_mask[:, :, :logits.shape[-2], :logits.shape[-1]].type_as(logits.data)
        weights = F.softmax(logits, dim=-1)
        weights = self.dropout(weights)
        contexts = torch.matmul(weights, values)
        contexts = self._merge_heads(contexts)
        outputs = self.output_linear(contexts)
        if self.attention_map:
            return outputs, weights
        return outputs

class self_attention_block(nn.Module):
    def __init__(self, hidden_size, total_key_depth, total_value_depth, filter_size, num_heads,
                 bias_mask=None, layer_dropout=0.0, attention_dropout=0.0, relu_dropout=0.0, attention_map=False):
        super().__init__()
        self.attention_map = attention_map
        self.multi_head_attention = MultiHeadAttention(
            hidden_size, total_key_depth, total_value_depth, hidden_size,
            num_heads, bias_mask, attention_dropout, attention_map)
        self.positionwise_convolution = PositionwiseFeedForward(
            hidden_size, filter_size, hidden_size, layer_config='cc', padding='both', dropout=relu_dropout)
        self.dropout = nn.Dropout(layer_dropout)
        self.layer_norm_mha = LayerNorm(hidden_size)
        self.layer_norm_ffn = LayerNorm(hidden_size)
    def forward(self, inputs):
        x = inputs
        x_norm = self.layer_norm_mha(x)
        if self.attention_map:
            y, weights = self.multi_head_attention(x_norm, x_norm, x_norm)
        else:
            y = self.multi_head_attention(x_norm, x_norm, x_norm)
        x = self.dropout(x + y)
        x_norm = self.layer_norm_ffn(x)
        y = self.positionwise_convolution(x_norm)
        y = self.dropout(x + y)
        if self.attention_map:
            return y, weights
        return y

class bi_directional_self_attention(nn.Module):
    def __init__(self, hidden_size, total_key_depth, total_value_depth, filter_size, num_heads, max_length,
                 layer_dropout=0.0, attention_dropout=0.0, relu_dropout=0.0):
        super().__init__()
        self.weights_list = list()
        params = (hidden_size, total_key_depth or hidden_size, total_value_depth or hidden_size,
                  filter_size, num_heads, _gen_bias_mask(max_length),
                  layer_dropout, attention_dropout, relu_dropout, True)
        self.attn_block = self_attention_block(*params)
        params = (hidden_size, total_key_depth or hidden_size, total_value_depth or hidden_size,
                  filter_size, num_heads, torch.transpose(_gen_bias_mask(max_length), dim0=2, dim1=3),
                  layer_dropout, attention_dropout, relu_dropout, True)
        self.backward_attn_block = self_attention_block(*params)
        self.linear = nn.Linear(hidden_size * 2, hidden_size)
    def forward(self, inputs):
        x, list_ = inputs
        encoder_outputs, weights = self.attn_block(x)
        reverse_outputs, reverse_weights = self.backward_attn_block(x)
        outputs = torch.cat((encoder_outputs, reverse_outputs), dim=2)
        y = self.linear(outputs)
        self.weights_list = list_
        self.weights_list.append(weights)
        self.weights_list.append(reverse_weights)
        return y, self.weights_list

class bi_directional_self_attention_layers(nn.Module):
    def __init__(self, embedding_size, hidden_size, num_layers, num_heads, total_key_depth, total_value_depth,
                 filter_size, max_length=100, input_dropout=0.0, layer_dropout=0.0,
                 attention_dropout=0.0, relu_dropout=0.0):
        super().__init__()
        self.timing_signal = _gen_timing_signal(max_length, hidden_size)
        params = (hidden_size, total_key_depth or hidden_size, total_value_depth or hidden_size,
                  filter_size, num_heads, max_length,
                  layer_dropout, attention_dropout, relu_dropout)
        self.embedding_proj = nn.Linear(embedding_size, hidden_size, bias=False)
        self.self_attn_layers = nn.Sequential(*[bi_directional_self_attention(*params) for _ in range(num_layers)])
        self.layer_norm = LayerNorm(hidden_size)
        self.input_dropout = nn.Dropout(input_dropout)
    def forward(self, inputs):
        x = self.input_dropout(inputs)
        x = self.embedding_proj(x)
        x += self.timing_signal[:, :inputs.shape[1], :].type_as(inputs.data)
        y, weights_list = self.self_attn_layers((x, []))
        y = self.layer_norm(y)
        return y, weights_list

class SoftmaxOutputLayer(nn.Module):
    def __init__(self, hidden_size, output_size, probs_out=False):
        super().__init__()
        self.output_size = output_size
        self.output_projection = nn.Linear(hidden_size, output_size)
        self.probs_out = probs_out
        self.lstm = nn.LSTM(input_size=hidden_size, hidden_size=int(hidden_size / 2),
                           batch_first=True, bidirectional=True)
        self.hidden_size = hidden_size
    def forward(self, hidden):
        logits = self.output_projection(hidden)
        probs = F.softmax(logits, -1)
        topk, indices = torch.topk(probs, 2)
        predictions = indices[:, :, 0]
        second = indices[:, :, 1]
        if self.probs_out:
            return logits
        return predictions, second
    def loss(self, hidden, labels):
        logits = self.output_projection(hidden)
        log_probs = F.log_softmax(logits, -1)
        return F.nll_loss(log_probs.view(-1, self.output_size), labels.view(-1))

class BTC_model(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.timestep = config['timestep']
        self.probs_out = config['probs_out']
        params = (config['feature_size'], config['hidden_size'], config['num_layers'],
                  config['num_heads'], config['total_key_depth'], config['total_value_depth'],
                  config['filter_size'], config['timestep'], config['input_dropout'],
                  config['layer_dropout'], config['attention_dropout'], config['relu_dropout'])
        self.self_attn_layers = bi_directional_self_attention_layers(*params)
        self.output_layer = SoftmaxOutputLayer(
            hidden_size=config['hidden_size'], output_size=config['num_chords'],
            probs_out=config['probs_out'])
    def forward(self, x, labels):
        labels = labels.view(-1, self.timestep)
        self_attn_output, weights_list = self.self_attn_layers(x)
        if self.probs_out:
            logits = self.output_layer(self_attn_output)
            return logits
        prediction, second = self.output_layer(self_attn_output)
        prediction = prediction.view(-1)
        second = second.view(-1)
        loss = self.output_layer.loss(self_attn_output, labels)
        return prediction, loss, weights_list, second

print('BTC model architecture defined.')
print(f'Model config: 8 bi-directional self-attention layers, 4 heads, hidden=128')
print(f'Input: CQT features (144 bins, 24 bins/octave, hop=2048, sr=22050)')
print(f'Output: {NUM_CHORDS} chord classes (12 roots x 14 qualities + N + X)')

## Cell 4: Training Data Preparation with Pitch Augmentation

In [ ]:
# ============================================================
# Cell 4: Training Data Preparation with Pitch Augmentation
# ============================================================
# For each training sample:
#   - Create 12 pitch-shifted copies (all semitone transpositions)
#   - Transpose chord labels accordingly
#   - Oversample underrepresented sharp/flat roots
# ============================================================

import json
from pathlib import Path
from collections import Counter

# BTC config constants
BTC_SR = 22050
BTC_N_BINS = 144
BTC_BINS_PER_OCTAVE = 24
BTC_HOP_LENGTH = 2048
BTC_INST_LEN = 10.0
N_TIMESTEP = 108  # from model config
TIME_UNIT = BTC_INST_LEN / N_TIMESTEP

def parse_lab_file(lab_path):
    """Parse a .lab file into list of (start, end, chord) tuples."""
    events = []
    with open(lab_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) >= 3:
                try:
                    start = float(parts[0])
                    end = float(parts[1])
                    chord = parts[2]
                    events.append((start, end, chord))
                except ValueError:
                    continue
    return events

def transpose_chord(chord_idx, semitones):
    """Transpose a chord index by N semitones.
    
    Chord index layout: root (0-11) * 14 + quality (0-13)
    Indices 168 (X) and 169 (N) are unchanged.
    """
    if chord_idx >= 168:  # N or X
        return chord_idx
    root = chord_idx // 14
    quality = chord_idx % 14
    new_root = (root + semitones) % 12
    return new_root * 14 + quality

def shift_cqt_features(feature_segment, semitones, bins_per_octave=24):
    """Shift CQT features by N semitones via circular rolling.
    
    CQT has `bins_per_octave` bins per octave, so shifting by 1 semitone
    = shifting by `bins_per_octave / 12` bins = 2 bins for 24 bins/octave.
    
    Args:
        feature_segment: shape [timesteps, n_bins] (e.g., [108, 144])
        semitones: integer number of semitones to shift
        bins_per_octave: CQT bins per octave (24 for BTC)
    
    Returns:
        Shifted feature segment of same shape
    """
    if semitones == 0:
        return feature_segment.copy()
    
    bins_per_semitone = bins_per_octave // 12  # = 2 for 24 bins/octave
    shift_bins = semitones * bins_per_semitone
    
    shifted = np.zeros_like(feature_segment)
    n_bins = feature_segment.shape[1]
    
    if shift_bins > 0:
        # Shift up: bins move to higher indices
        if shift_bins < n_bins:
            shifted[:, shift_bins:] = feature_segment[:, :n_bins - shift_bins]
    else:
        # Shift down: bins move to lower indices
        abs_shift = abs(shift_bins)
        if abs_shift < n_bins:
            shifted[:, :n_bins - abs_shift] = feature_segment[:, abs_shift:]
    
    return shifted

# Load checkpoint to get mean/std for normalization
print('Loading checkpoint for mean/std...')
checkpoint_path = f'{WORK_DIR}/btc_finetuned_best.pt'
torch.serialization.add_safe_globals([np._core.multiarray.scalar, np.ndarray, np.dtype, np.dtypes.Float64DType])
ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
mean = ckpt['mean']
std = ckpt['std']
print(f'  mean shape: {mean.shape}, std shape: {std.shape}')

# Load manifest
manifest_path = f'{WORK_DIR}/manifest.json'
if os.path.exists(manifest_path):
    with open(manifest_path) as f:
        manifest = json.load(f)
    song_ids = manifest['songs']
    print(f'Manifest: {len(song_ids)} songs')
else:
    # Build from available files
    feat_files = sorted(Path(f'{WORK_DIR}/features').glob('*.npy'))
    lab_files = sorted(Path(f'{WORK_DIR}/labels').glob('*.lab'))
    feat_ids = {f.stem for f in feat_files}
    lab_ids = {f.stem for f in lab_files}
    song_ids = sorted(feat_ids & lab_ids)
    print(f'Found {len(song_ids)} paired feature/label files')

# Build training data with pitch augmentation
print('\nBuilding training data with pitch augmentation...')
print('For each song segment: 12 transpositions (all semitones)\n')

all_features = []
all_labels = []
root_counts = Counter()
skipped = 0

for i, song_id in enumerate(song_ids):
    feat_path = f'{WORK_DIR}/features/{song_id}.npy'
    lab_path = f'{WORK_DIR}/labels/{song_id}.lab'
    
    if not os.path.exists(feat_path) or not os.path.exists(lab_path):
        skipped += 1
        continue
    
    try:
        # Load CQT features [n_bins, n_frames] -> [n_frames, n_bins]
        feature = np.load(feat_path).T
        feature = (feature - mean) / std
        
        # Parse labels
        events = parse_lab_file(lab_path)
        if not events:
            skipped += 1
            continue
        
        # Create frame-level labels
        n_frames = feature.shape[0]
        n_idx = CHORD_TO_IDX.get('N', 169)
        frame_labels = np.full(n_frames, n_idx, dtype=np.int64)
        
        for start, end, chord in events:
            chord_idx = CHORD_TO_IDX.get(chord)
            if chord_idx is None:
                continue
            start_frame = max(0, min(int(start / TIME_UNIT), n_frames - 1))
            end_frame = max(0, min(int(end / TIME_UNIT), n_frames))
            frame_labels[start_frame:end_frame] = chord_idx
        
        # Pad to multiple of N_TIMESTEP
        num_pad = N_TIMESTEP - (n_frames % N_TIMESTEP)
        if num_pad < N_TIMESTEP:
            feature = np.pad(feature, ((0, num_pad), (0, 0)), mode='constant')
            frame_labels = np.pad(frame_labels, (0, num_pad), mode='constant', constant_values=n_idx)
        
        n_segments = feature.shape[0] // N_TIMESTEP
        
        # For each segment, create 12 transpositions
        for seg in range(n_segments):
            s = seg * N_TIMESTEP
            e = s + N_TIMESTEP
            seg_feat = feature[s:e]   # [108, 144]
            seg_lab = frame_labels[s:e]  # [108]
            
            # Skip segments that are all N (silence)
            if np.all(seg_lab == n_idx):
                continue
            
            for shift in range(12):  # 0 to 11 semitones
                shifted_feat = shift_cqt_features(seg_feat, shift)
                shifted_lab = np.array([transpose_chord(int(c), shift) for c in seg_lab], dtype=np.int64)
                
                all_features.append(shifted_feat)
                all_labels.append(shifted_lab)
                
                # Count roots for balance tracking
                for c in shifted_lab:
                    if c < 168:
                        root_counts[ROOT_LIST[c // 14]] += 1
    
    except Exception as ex:
        skipped += 1
        if i < 5:
            print(f'  Skipped {song_id}: {ex}')
    
    if (i + 1) % 200 == 0:
        print(f'  Processed {i + 1}/{len(song_ids)} songs, {len(all_features)} segments so far...')

print(f'\nProcessed {len(song_ids) - skipped} songs, skipped {skipped}')
print(f'Total augmented segments: {len(all_features)}')

# Convert to arrays
X = np.array(all_features, dtype=np.float32)
Y = np.array(all_labels, dtype=np.int64)
print(f'X shape: {X.shape}')  # [N, 108, 144]
print(f'Y shape: {Y.shape}')  # [N, 108]

# Root distribution analysis
print('\nRoot distribution after augmentation:')
total_root = sum(root_counts.values())
for root in ROOT_LIST:
    count = root_counts[root]
    pct = 100 * count / total_root if total_root > 0 else 0
    bar = '#' * int(pct * 3)
    print(f'  {root:>2s}: {count:>8d} ({pct:5.1f}%) {bar}')

# Oversample underrepresented roots (sharp/flat)
sharp_flat_roots = {'C#', 'D#', 'F#', 'G#', 'A#'}
natural_roots = {'C', 'D', 'E', 'F', 'G', 'A', 'B'}

sharp_count = sum(root_counts[r] for r in sharp_flat_roots)
natural_count = sum(root_counts[r] for r in natural_roots)

if natural_count > 0 and sharp_count > 0:
    ratio = natural_count / sharp_count
    print(f'\nNatural/Sharp ratio: {ratio:.2f}x')
    
    if ratio > 1.5:
        # Oversample sharp/flat segments
        print('Oversampling sharp/flat chord segments...')
        sharp_indices = []
        for idx in range(len(Y)):
            roots_in_seg = set()
            for c in Y[idx]:
                if c < 168:
                    roots_in_seg.add(ROOT_LIST[c // 14])
            if roots_in_seg & sharp_flat_roots:
                sharp_indices.append(idx)
        
        # Duplicate sharp/flat segments to balance
        n_oversample = min(len(sharp_indices), int(len(X) * 0.2))  # Cap at 20% extra
        if n_oversample > 0:
            oversample_idx = np.random.choice(sharp_indices, n_oversample, replace=True)
            X = np.concatenate([X, X[oversample_idx]], axis=0)
            Y = np.concatenate([Y, Y[oversample_idx]], axis=0)
            print(f'  Added {n_oversample} oversampled segments')
            print(f'  Final dataset: {len(X)} segments')
    else:
        print('Root distribution already balanced (ratio <= 1.5)')

# Shuffle
perm = np.random.permutation(len(X))
X = X[perm]
Y = Y[perm]

# Train/val split (90/10)
n_val = max(1, int(len(X) * 0.1))
X_val, Y_val = X[:n_val], Y[:n_val]
X_train, Y_train = X[n_val:], Y[n_val:]

print(f'\nTrain: {len(X_train)} segments')
print(f'Val:   {len(X_val)} segments')
print(f'\nMemory usage: {X.nbytes / 1e9:.2f} GB')
print('Data preparation complete.')

## Cell 5: Training Loop

In [ ]:
# ============================================================
# Cell 5: Training Loop — Fine-tune BTC on augmented data
# ============================================================

import time

# Load config
config_path = f'{WORK_DIR}/run_config.yaml'
config = HParams.load(config_path)
config.feature['large_voca'] = True
config.model['num_chords'] = NUM_CHORDS

# Build model
model = BTC_model(config=config.model).to(device)

# Load existing fine-tuned checkpoint (transfer learning)
print('Loading existing fine-tuned checkpoint...')
ckpt = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(ckpt['model'])
mean_tensor = ckpt['mean']
std_tensor = ckpt['std']

prev_epoch = ckpt.get('epoch', 0)
prev_val_loss = ckpt.get('val_loss', 'N/A')
prev_val_acc = ckpt.get('val_acc', 'N/A')
print(f'  Previous training: epoch {prev_epoch}, val_loss={prev_val_loss}, val_acc={prev_val_acc}')

n_params = sum(p.numel() for p in model.parameters())
print(f'  Model parameters: {n_params:,}')

# Training hyperparameters
LEARNING_RATE = 5e-5     # Lower LR for fine-tuning (was 1e-4)
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 64          # Larger batch for GPU
MAX_EPOCHS = 80
PATIENCE = 15
GRAD_CLIP = 1.0

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, min_lr=1e-6)

# Class weights to handle imbalance (downweight N, upweight rare chords)
all_labels_flat = Y_train.flatten()
class_counts = np.bincount(all_labels_flat, minlength=NUM_CHORDS).astype(float)
class_counts[class_counts == 0] = 1
weights = 1.0 / np.sqrt(class_counts)  # sqrt for softer weighting
weights = weights / weights.sum() * NUM_CHORDS
# Extra downweight for N class (index 169)
weights[169] *= 0.3
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

best_val_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'lr': []}

save_dir = f'{WORK_DIR}/checkpoints'
os.makedirs(save_dir, exist_ok=True)

print(f'\nTraining config:')
print(f'  LR: {LEARNING_RATE}, Batch: {BATCH_SIZE}, Max epochs: {MAX_EPOCHS}')
print(f'  Train segments: {len(X_train)}, Val segments: {len(X_val)}')
print(f'  Device: {device}')
print(f'\n{"="*80}')
print(f'{"Epoch":>5s} | {"Train Loss":>10s} | {"Val Loss":>10s} | {"Val Acc":>8s} | {"LR":>10s} | {"Time":>6s}')
print(f'{"="*80}')

for epoch in range(MAX_EPOCHS):
    epoch_start = time.time()
    
    # --- Train ---
    model.train()
    train_losses = []
    perm = np.random.permutation(len(X_train))
    
    for batch_start in range(0, len(X_train), BATCH_SIZE):
        batch_idx = perm[batch_start:batch_start + BATCH_SIZE]
        x_batch = torch.tensor(X_train[batch_idx]).to(device)
        y_batch = torch.tensor(Y_train[batch_idx]).to(device)
        
        # Forward through BTC encoder
        encoder_output, _ = model.self_attn_layers(x_batch)
        
        # Compute loss using output layer's projection
        logits = model.output_layer.output_projection(encoder_output)
        loss = criterion(logits.view(-1, NUM_CHORDS), y_batch.view(-1))
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        train_losses.append(loss.item())
    
    avg_train_loss = np.mean(train_losses)
    
    # --- Validate ---
    model.eval()
    val_losses = []
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_start in range(0, len(X_val), BATCH_SIZE):
            x_batch = torch.tensor(X_val[batch_start:batch_start + BATCH_SIZE]).to(device)
            y_batch = torch.tensor(Y_val[batch_start:batch_start + BATCH_SIZE]).to(device)
            
            encoder_output, _ = model.self_attn_layers(x_batch)
            logits = model.output_layer.output_projection(encoder_output)
            loss = criterion(logits.view(-1, NUM_CHORDS), y_batch.view(-1))
            val_losses.append(loss.item())
            
            preds = logits.argmax(dim=-1)
            # Only count non-N frames for accuracy
            mask = y_batch != 169
            correct += (preds[mask] == y_batch[mask]).sum().item()
            total += mask.sum().item()
    
    avg_val_loss = np.mean(val_losses)
    val_acc = correct / total if total > 0 else 0
    lr = optimizer.param_groups[0]['lr']
    elapsed = time.time() - epoch_start
    
    scheduler.step(avg_val_loss)
    
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(lr)
    
    marker = ''
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        marker = ' *BEST*'
        
        # Save best checkpoint
        save_path = f'{save_dir}/btc_finetuned_best.pt'
        torch.save({
            'model': model.state_dict(),
            'mean': mean_tensor,
            'std': std_tensor,
            'epoch': epoch + 1,
            'val_loss': avg_val_loss,
            'val_acc': val_acc,
            'train_segments': len(X_train),
            'augmentation': '12_semitone_transpositions_with_oversampling',
        }, save_path)
        
        # Also save to Drive
        drive_save = f'{DRIVE_DATA}/btc_finetuned_best.pt'
        torch.save({
            'model': model.state_dict(),
            'mean': mean_tensor,
            'std': std_tensor,
            'epoch': epoch + 1,
            'val_loss': avg_val_loss,
            'val_acc': val_acc,
            'train_segments': len(X_train),
            'augmentation': '12_semitone_transpositions_with_oversampling',
        }, drive_save)
    else:
        patience_counter += 1
    
    print(f'{epoch+1:5d} | {avg_train_loss:10.4f} | {avg_val_loss:10.4f} | {val_acc:7.4f} | {lr:10.2e} | {elapsed:5.1f}s{marker}')
    
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch + 1} (no improvement for {PATIENCE} epochs)')
        break

print(f'\n{"="*80}')
print(f'Training complete. Best val_loss: {best_val_loss:.4f}')
print(f'Best checkpoint saved to: {save_dir}/btc_finetuned_best.pt')
print(f'Also saved to Google Drive: {DRIVE_DATA}/btc_finetuned_best.pt')

In [ ]:
# --- Training curves ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['val_acc'])
axes[1].set_title('Validation Accuracy (non-N frames)')
axes[1].set_xlabel('Epoch')
axes[1].grid(True)

axes[2].plot(history['lr'])
axes[2].set_title('Learning Rate')
axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log')
axes[2].grid(True)

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/training_curves.png', dpi=150)
plt.show()
print(f'Training curves saved to {WORK_DIR}/training_curves.png')

## Cell 6: Evaluation

In [ ]:
# ============================================================
# Cell 6: Evaluation
# ============================================================
# Test on "The Time Comes" ground truth:
#   F#m, A, Bm, E, B, D, C#m, G (key of F#m / A major)
#
# Also: confusion matrix for chord roots, before/after comparison
# ============================================================

import matplotlib.pyplot as plt
import librosa

def mir_to_simple(chord):
    """Convert mir_eval chord format (C:min7) to simple format (Cm7)."""
    if chord in ('N', 'X'):
        return 'N'
    return (chord
            .replace(':minmaj7', 'mMaj7')
            .replace(':min7', 'm7')
            .replace(':min6', 'm6')
            .replace(':min', 'm')
            .replace(':maj7', 'maj7')
            .replace(':maj6', '6')
            .replace(':maj', '')
            .replace(':7', '7')
            .replace(':aug', 'aug')
            .replace(':dim7', 'dim7')
            .replace(':dim', 'dim')
            .replace(':sus4', 'sus4')
            .replace(':sus2', 'sus2')
            .replace(':hdim7', 'hdim7'))

def run_btc_inference(model, audio_path, mean, std, config, device):
    """Run BTC chord detection on an audio file. Returns list of (chord_name, duration)."""
    model.eval()
    
    sr = config.mp3['song_hz']
    wav, _ = librosa.load(audio_path, sr=sr, mono=True)
    duration = len(wav) / sr
    
    # Tuning compensation
    tuning_offset = librosa.estimate_tuning(y=wav, sr=sr)
    if abs(tuning_offset) > 0.05:
        wav = librosa.effects.pitch_shift(wav, sr=sr, n_steps=-tuning_offset)
        print(f'  Applied tuning compensation: {-tuning_offset:+.3f} semitones')
    
    # CQT features
    inst_len = config.mp3['inst_len']
    feature = None
    current = 0
    while len(wav) > current + int(sr * inst_len):
        start_idx = int(current)
        end_idx = int(current + sr * inst_len)
        tmp = librosa.cqt(wav[start_idx:end_idx], sr=sr,
                         n_bins=config.feature['n_bins'],
                         bins_per_octave=config.feature['bins_per_octave'],
                         hop_length=config.feature['hop_length'])
        feature = tmp if feature is None else np.concatenate((feature, tmp), axis=1)
        current = end_idx
    tmp = librosa.cqt(wav[int(current):], sr=sr,
                     n_bins=config.feature['n_bins'],
                     bins_per_octave=config.feature['bins_per_octave'],
                     hop_length=config.feature['hop_length'])
    feature = tmp if feature is None else np.concatenate((feature, tmp), axis=1)
    feature = np.log(np.abs(feature) + 1e-6)
    
    feature = feature.T
    feature = (feature - mean) / std
    
    n_timestep = config.model['timestep']
    time_unit = inst_len / n_timestep
    num_pad = n_timestep - (feature.shape[0] % n_timestep)
    feature = np.pad(feature, ((0, num_pad), (0, 0)), mode='constant')
    num_instance = feature.shape[0] // n_timestep
    
    events = []
    start_time = 0.0
    prev_chord = None
    
    with torch.no_grad():
        feat_tensor = torch.tensor(feature, dtype=torch.float32).unsqueeze(0).to(device)
        for t in range(num_instance):
            chunk = feat_tensor[:, n_timestep * t:n_timestep * (t + 1), :]
            encoder_output, _ = model.self_attn_layers(chunk)
            # Use LSTM path (same as production code)
            lstm_out, _ = model.output_layer.lstm(encoder_output)
            logits = model.output_layer.output_projection(lstm_out).squeeze(0)
            prediction = logits.argmax(dim=-1)
            
            for i in range(n_timestep):
                current_time = time_unit * (n_timestep * t + i)
                if current_time > duration:
                    break
                chord_idx = prediction[i].item()
                if prev_chord is None:
                    prev_chord = chord_idx
                    start_time = current_time
                    continue
                if chord_idx != prev_chord:
                    chord_name = mir_to_simple(IDX_TO_CHORD[prev_chord])
                    dur = current_time - start_time
                    if dur >= 0.3 and chord_name != 'N':
                        events.append((chord_name, dur, start_time))
                    start_time = current_time
                    prev_chord = chord_idx
        
        if prev_chord is not None:
            chord_name = mir_to_simple(IDX_TO_CHORD[prev_chord])
            dur = duration - start_time
            if dur >= 0.3 and chord_name != 'N':
                events.append((chord_name, dur, start_time))
    
    return events

# --- Evaluation on validation set: Root confusion matrix ---
print('Computing root confusion matrix on validation set...')

model.eval()
all_true_roots = []
all_pred_roots = []

with torch.no_grad():
    for batch_start in range(0, len(X_val), BATCH_SIZE):
        x_batch = torch.tensor(X_val[batch_start:batch_start + BATCH_SIZE]).to(device)
        y_batch = Y_val[batch_start:batch_start + BATCH_SIZE]
        
        encoder_output, _ = model.self_attn_layers(x_batch)
        logits = model.output_layer.output_projection(encoder_output)
        preds = logits.argmax(dim=-1).cpu().numpy()
        
        for seg_idx in range(len(y_batch)):
            for frame_idx in range(N_TIMESTEP):
                true_c = y_batch[seg_idx, frame_idx]
                pred_c = preds[seg_idx, frame_idx]
                if true_c < 168 and pred_c < 168:  # Exclude N and X
                    all_true_roots.append(true_c // 14)
                    all_pred_roots.append(pred_c // 14)

# Build confusion matrix
confusion = np.zeros((12, 12), dtype=np.int64)
for t, p in zip(all_true_roots, all_pred_roots):
    confusion[t, p] += 1

# Normalize by row (true label)
row_sums = confusion.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
confusion_norm = confusion / row_sums

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
im = ax.imshow(confusion_norm, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(12))
ax.set_yticks(range(12))
ax.set_xticklabels(ROOT_LIST)
ax.set_yticklabels(ROOT_LIST)
ax.set_xlabel('Predicted Root')
ax.set_ylabel('True Root')
ax.set_title('Root Confusion Matrix (Normalized by Row)')

for i in range(12):
    for j in range(12):
        val = confusion_norm[i, j]
        color = 'white' if val > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=8)

plt.colorbar(im)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/root_confusion_matrix.png', dpi=150)
plt.show()

# Per-root accuracy
print('\nPer-root accuracy:')
for i, root in enumerate(ROOT_LIST):
    acc = confusion_norm[i, i]
    bar = '#' * int(acc * 40)
    print(f'  {root:>2s}: {acc:.3f} {bar}')

overall_acc = np.trace(confusion) / confusion.sum()
print(f'\nOverall root accuracy: {overall_acc:.4f}')

# Sharp/flat vs natural accuracy
sharp_indices = [1, 3, 6, 8, 10]  # C#, D#, F#, G#, A#
natural_indices = [0, 2, 4, 5, 7, 9, 11]  # C, D, E, F, G, A, B
sharp_correct = sum(confusion[i, i] for i in sharp_indices)
sharp_total = sum(confusion[i, :].sum() for i in sharp_indices)
natural_correct = sum(confusion[i, i] for i in natural_indices)
natural_total = sum(confusion[i, :].sum() for i in natural_indices)

print(f'\nSharp/flat root accuracy: {sharp_correct / max(sharp_total, 1):.4f} ({sharp_correct}/{sharp_total})')
print(f'Natural root accuracy:    {natural_correct / max(natural_total, 1):.4f} ({natural_correct}/{natural_total})')

In [ ]:
# ============================================================
# Test on "The Time Comes" (if audio available)
# ============================================================
# Ground truth: F#m, A, Bm, E, B, D, C#m, G — key of F#m
#
# Upload the audio file or provide a path below.
# If no audio available, this cell shows the expected vs model vocabulary.
# ============================================================

THE_TIME_COMES_GROUND_TRUTH = ['F#m', 'A', 'Bm', 'E', 'B', 'D', 'C#m', 'G']
THE_TIME_COMES_KEY = 'F#m'

# Try to find test audio
test_audio = None
possible_paths = [
    f'{WORK_DIR}/test_audio/the_time_comes.wav',
    f'{DRIVE_DATA}/test_audio/the_time_comes.wav',
    '/content/the_time_comes.wav',
]
for p in possible_paths:
    if os.path.exists(p):
        test_audio = p
        break

if test_audio is None:
    print('No test audio found for "The Time Comes".')
    print('To test, upload the audio file:')
    try:
        from google.colab import files
        print('  Upload now (or skip this cell):')
        uploaded = files.upload()
        for name in uploaded:
            test_audio = f'/content/{name}'
            break
    except:
        pass

if test_audio:
    print(f'\nRunning inference on: {test_audio}')
    print(f'Ground truth chords: {THE_TIME_COMES_GROUND_TRUTH}')
    print(f'Ground truth key: {THE_TIME_COMES_KEY}')
    print()
    
    # Run with retrained model
    events = run_btc_inference(model, test_audio, mean_tensor, std_tensor, config, device)
    
    print('Detected chords:')
    detected_unique = set()
    for chord, dur, t in events:
        print(f'  {t:6.1f}s - {t+dur:6.1f}s ({dur:5.1f}s): {chord}')
        detected_unique.add(chord.rstrip('0123456789'))
    
    # Score: how many ground truth chords appear in detection
    detected_roots = set()
    for chord, dur, t in events:
        # Extract root
        root = chord[0]
        if len(chord) > 1 and chord[1] in ('#', 'b'):
            root = chord[:2]
        detected_roots.add(root)
    
    gt_roots = set()
    for chord in THE_TIME_COMES_GROUND_TRUTH:
        root = chord[0]
        if len(chord) > 1 and chord[1] in ('#', 'b'):
            root = chord[:2]
        gt_roots.add(root)
    
    matched = gt_roots & detected_roots
    print(f'\nGround truth roots: {sorted(gt_roots)}')
    print(f'Detected roots:     {sorted(detected_roots)}')
    print(f'Matched: {len(matched)}/{len(gt_roots)} ({100*len(matched)/len(gt_roots):.0f}%): {sorted(matched)}')
    missing = gt_roots - detected_roots
    if missing:
        print(f'Missing: {sorted(missing)}')
    extra = detected_roots - gt_roots
    if extra:
        print(f'Extra (not in ground truth): {sorted(extra)}')
else:
    print('\nSkipping audio test. To evaluate, upload "The Time Comes" audio.')
    print(f'\nGround truth vocabulary check:')
    print(f'  Expected chords: {THE_TIME_COMES_GROUND_TRUTH}')
    print(f'  Model vocabulary coverage:')
    for chord in THE_TIME_COMES_GROUND_TRUTH:
        # Map to Harte notation
        if chord.endswith('m') and not chord.endswith('dim'):
            root = chord[:-1]
            harte = f'{root}:min'
        else:
            harte = chord
        idx = CHORD_TO_IDX.get(harte, CHORD_TO_IDX.get(chord, None))
        status = f'idx={idx}' if idx is not None else 'NOT FOUND'
        print(f'    {chord:>4s} -> {harte:>6s} -> {status}')

## Cell 7: Download Retrained Checkpoint

In [ ]:
# ============================================================
# Cell 7: Download retrained checkpoint
# ============================================================

from google.colab import files

best_ckpt = f'{save_dir}/btc_finetuned_best.pt'

if os.path.exists(best_ckpt):
    size_mb = os.path.getsize(best_ckpt) / 1e6
    print(f'Best checkpoint: {best_ckpt} ({size_mb:.1f} MB)')
    
    # Verify checkpoint contents
    ckpt_info = torch.load(best_ckpt, map_location='cpu', weights_only=True)
    print(f'  Epoch: {ckpt_info.get("epoch", "?")}')
    print(f'  Val loss: {ckpt_info.get("val_loss", "?")}')
    print(f'  Val acc: {ckpt_info.get("val_acc", "?")}')
    print(f'  Training segments: {ckpt_info.get("train_segments", "?")}')
    print(f'  Augmentation: {ckpt_info.get("augmentation", "?")}')
    
    print(f'\nCheckpoint also saved to Google Drive: {DRIVE_DATA}/btc_finetuned_best.pt')
    print(f'\nTo use locally, copy to:')
    print(f'  ~/stemscribe/backend/training_data/btc_finetune/checkpoints/btc_finetuned_best.pt')
    
    # Download to browser
    print('\nStarting download...')
    files.download(best_ckpt)
else:
    print('No checkpoint found. Run the training cell first.')